# 🏁 Aula 20 — Definição do Projeto Final

**Curso:** Técnico em Inteligência Artificial | **UC:** Introdução a Arquitetura de Computadores

---

## 🎯 Situação de Aprendizagem

A empresa organizou um **hackathon interno de 4 semanas**: cada time deve desenvolver uma solução de IA acelerada por GPU para um problema real. Os times têm acesso a um servidor com **2× RTX 4090** e ao **Google Colab Pro**. O desafio desta aula é estruturar o projeto do zero: escolher o domínio (visão computacional, PNL ou séries temporais), identificar o problema, selecionar o dataset e planejar a arquitetura — integrando as técnicas de monitoramento e automação aprendidas nas aulas anteriores.

Este notebook funciona **sem GPU** (modo simulado). O objetivo é planejar, não treinar.

In [ ]:
# 1. Verificar o ambiente disponível (GPU real ou modo simulado)
import shutil

try:
    import torch
    TEM_TORCH = True
    GPU = torch.cuda.is_available()
except ImportError:
    TEM_TORCH = False
    GPU = False

TEM_NVIDIA_SMI = shutil.which("nvidia-smi") is not None

print(f"PyTorch instalado : {TEM_TORCH}")
print(f"nvidia-smi        : {TEM_NVIDIA_SMI}")
print(f"GPU CUDA           : {GPU}")
if GPU:
    print(f"  -> {torch.cuda.get_device_name(0)}")
    print(f"  -> VRAM: {torch.cuda.get_device_properties(0).total_memory // 1024**3} GB")
else:
    print("  -> Sem GPU: seguimos em MODO SIMULADO (o projeto é planejado, não treinado aqui).")

## 1. Escolhendo o domínio

Use este mapa para registrar o domínio do seu time. A escolha define o modelo, o dataset e o perfil de VRAM.

In [ ]:
# 2. Catálogo de domínios: modelos, datasets e perfil de GPU
DOMINIOS = {
    "visao": {
        "descricao": "Visão Computacional (imagens)",
        "exemplos": ["doenças em plantas", "contagem de células", "defeitos industriais"],
        "modelos": ["ResNet", "EfficientNet", "YOLO", "ViT"],
        "dataset_hf": "plants-village/PlantVillage",
        "perfil_gpu": "Alta VRAM (batches grandes de imagens)",
    },
    "pnl": {
        "descricao": "Processamento de Linguagem Natural (texto)",
        "exemplos": ["classificação de sentimentos", "extração de entidades", "sumarização"],
        "modelos": ["BERT", "RoBERTa", "GPT-2 fine-tune"],
        "dataset_hf": "imdb",
        "perfil_gpu": "VRAM e RAM balanceadas",
    },
    "series_temporais": {
        "descricao": "Séries Temporais",
        "exemplos": ["consumo elétrico", "anomalias IoT", "falhas de equipamento"],
        "modelos": ["LSTM", "Transformer", "TCN"],
        "dataset_hf": "ETDataset/ett",
        "perfil_gpu": "Menor VRAM, muitos steps",
    },
}

# Escolha o domínio do seu time (altere aqui):
MEU_DOMINIO = "visao"

info = DOMINIOS[MEU_DOMINIO]
print(f"Domínio escolhido: {info['descricao']}")
print(f"Exemplos          : {', '.join(info['exemplos'])}")
print(f"Modelos típicos   : {', '.join(info['modelos'])}")
print(f"Perfil de GPU     : {info['perfil_gpu']}")

## 2. Explorando datasets no Hugging Face

O Hugging Face reúne +70 mil datasets prontos. Abaixo tentamos carregar o dataset do domínio escolhido; se não houver internet/pacote, mostramos as **instruções equivalentes**.

In [ ]:
# 3. Tentar carregar o dataset do domínio (com fallback didático)
def explorar_dataset(chave_dominio, amostras=500):
    info = DOMINIOS.chave_dominio if False else DOMINIOS[chave_dominio]
    nome = info["dataset_hf"]
    try:
        from datasets import load_dataset
        ds = load_dataset(nome, split=f"train[:{amostras}]")
        print(f"Dataset '{nome}' carregado: {len(ds)} amostras")
        print(f"Colunas: {ds.column_names}")
        return ds
    except Exception as e:
        print(f"Não foi possível carregar '{nome}' agora ({type(e).__name__}).")
        print("Dica: execute `pip install datasets` e confirme a conexão de rede.")
        print(f"Código equivalente: load_dataset('{nome}', split='train[:{amostras}]')")
        return None

ds = explorar_dataset(MEU_DOMINIO)

## 3. Configuração central do projeto (dataclass)

Concentrar hiperparâmetros em um único lugar evita erros e facilita a reprodução.

In [ ]:
# 4. ConfigProjeto: todos os hiperparâmetros em um só lugar
from dataclasses import dataclass, asdict

@dataclass
class ConfigProjeto:
    nome: str = "MeuProjeto"
    dominio: str = "visao"          # visao | pnl | series_temporais
    device: str = "cuda" if GPU else "cpu"
    precision: str = "fp16"         # fp32 | fp16 | bf16
    batch_size: int = 32
    epochs: int = 10
    lr: float = 1e-3
    weight_decay: float = 1e-4
    num_classes: int = 38
    input_size: int = 224
    save_dir: str = "./checkpoints"
    log_dir: str = "./logs"

cfg = ConfigProjeto(nome="DeteccaoDoencas", dominio=MEU_DOMINIO, num_classes=38)
for k, v in asdict(cfg).items():
    print(f"  {k:<14} = {v}")

In [ ]:
# 5. Estimar a VRAM do projeto (regra de bolso) para planejar o hardware
PERFIL_VRAM_MB = {  # VRAM aproximada por amostra, por domínio (referência)
    "visao": 6.0,
    "pnl": 2.0,
    "series_temporais": 0.5,
}

def estimar_vram(cfg, dominio):
    base = 1500                       # overhead do runtime CUDA (MB)
    por_amostra = PERFIL_VRAM_MB[dominio]
    # fp16 usa ~metade da VRAM das ativações
    fator = 0.5 if cfg.precision in ("fp16", "bf16") else 1.0
    uso = base + cfg.batch_size * por_amostra * fator
    return uso

uso_fp32 = estimar_vram(ConfigProjeto(precision="fp32", batch_size=cfg.batch_size), MEU_DOMINIO)
uso_fp16 = estimar_vram(cfg, MEU_DOMINIO)
print(f"VRAM estimada (batch={cfg.batch_size}):")
print(f"  fp32: {uso_fp32:.0f} MB")
print(f"  {cfg.precision}: {uso_fp16:.0f} MB  (economia de {100*(1-uso_fp16/uso_fp32):.0f}%)")

for gpu, vram in [("RTX 4090", 24576), ("T4 (Colab)", 15360), ("V100 (Colab Pro)", 16384)]:
    print(f"  {'cabe' if uso_fp16 < vram else 'NAO cabe'} em {gpu} ({vram//1024} GB)")

## 4. Checklist de prontidão

Antes de implementar (Aula 21), o time precisa fechar este checklist. Marque cada item como `True` quando estiver resolvido.

In [ ]:
# 6. Checklist de prontidão do projeto
checklist = {
    "1. Definição do problema": {
        "Domínio escolhido (visão / PNL / séries temporais)": False,
        "Problema real identificado com impacto mensurável": False,
        "Dataset público disponível e compatível com Colab": False,
    },
    "2. Arquitetura técnica": {
        "Framework definido (PyTorch / TensorFlow)": False,
        "Modelo base escolhido (pré-treinado ou custom)": False,
        "Estratégia de GPU: CUDA / ROCm / Colab T4": False,
        "Precision: fp16 / bf16 habilitado para VRAM": False,
    },
    "3. Automação e monitoramento": {
        "Script de monitoramento GPU integrado (Aula 14)": False,
        "Fila de jobs configurada se múltiplas GPUs": False,
        "W&B ou MLflow configurado para logging": False,
    },
    "4. Repositório e documentação": {
        "Estrutura de pastas criada": False,
        "README com objetivo e instruções de reprodução": False,
        "requirements.txt completo": False,
    },
    "5. Critérios de avaliação": {
        "Métrica principal definida (acc / F1 / RMSE)": False,
        "Baseline simples implementado para comparação": False,
        "Meta de desempenho estabelecida": False,
    },
}

for categoria, itens in checklist.items():
    concluidos = sum(itens.values())
    print(f"\n{categoria} [{concluidos}/{len(itens)}]")
    for item, status in itens.items():
        print(f"  {'✅' if status else '⬜'} {item}")

pct = sum(v for c in checklist.values() for v in c.values()) / sum(len(c) for c in checklist.values())
print(f"\nProntidão geral: {pct*100:.0f}%")

---
## 📝 5. Exercícios Práticos (5)

Resolva os 5 exercícios a seguir completando as células de código correspondentes.

### Exercício 1: Definir o domínio do seu time

Altere `MEU_DOMINIO` para o domínio escolhido, crie seu `ConfigProjeto` e imprima todos os campos.

In [ ]:
# Escreva a sua solução para o Exercício 1 abaixo:
# MEU_DOMINIO = "pnl"  ... cfg = ConfigProjeto(...) ... print(asdict(cfg))

### Exercício 2: Estimar a VRAM de dois batch sizes

Use `estimar_vram` para comparar `batch_size=16` e `batch_size=64` no seu domínio com `fp16`. Qual cabe melhor na RTX 4090?

In [ ]:
# Escreva a sua solução para o Exercício 2 abaixo:
# for bs in (16, 64): cfg_tmp = ConfigProjeto(batch_size=bs) ...

### Exercício 3: Adicionar uma categoria ao checklist

Crie uma categoria **"6. Riscos"** com 2 itens (ex.: "Dataset desbalanceado tratado", "Plano B se a GPU falhar") e reimprima a prontidão geral.

In [ ]:
# Escreva a sua solução para o Exercício 3 abaixo:
# checklist["6. Riscos"] = {"...": False, ...}

### Exercício 4: Gerar o esqueleto de pastas do repositório

Escreva um trecho Python que **imprima** a árvore de diretórios do projeto (`config/`, `data/`, `models/`, `training/`, `scripts/`, `notebooks/`, `checkpoints/`).

In [ ]:
# Escreva a sua solução para o Exercício 4 abaixo:
# pastas = [...] ; for p in pastas: print(p)

### Exercício 5: Definir o baseline

Para o seu domínio, escreva qual será o **baseline mais simples** (ex.: classificador majoritário, regressão logística, média móvel) e qual métrica principal você vai usar. Imprima uma frase de resumo.

In [ ]:
# Escreva a sua solução para o Exercício 5 abaixo:
# baseline = "..."; metrica = "..."; print(f"Baseline: {baseline} | Métrica: {metrica}")

---
## 📌 6. Síntese e Conclusão

- **Transfer Learning:** reutilizar pesos pré-treinados reduz em ~10× o tempo de convergência.
- **Mixed Precision:** fp16/bf16 reduz a VRAM à metade e aumenta o throughput em 30–50%.
- **W&B / MLflow:** rastreiam hiperparâmetros, métricas e estatísticas de GPU de cada experimento.
- **DataLoader tuning:** `num_workers` + `pin_memory` eliminam o gargalo CPU→GPU.
- **Baseline primeiro:** implemente a solução simples antes do modelo complexo.
- **Estrutura de repositório:** separe config, data, models e scripts desde o início.

**Tarefa para a Aula 21:** repositório criado, dataset carregando no Colab e **baseline medido**.